# Настройка DuckLake

In [1]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = True
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 5
%config SqlMagic.named_parameters = "enabled"

In [2]:
import duckdb
import pandas as pd

%load_ext sql
conn = duckdb.connect()
%sql conn --alias duckdb

Tip: You may define configurations in /Users/i.korsakov/_code/github/pet_project_what_is_ducklake/pyproject.toml or /Users/i.korsakov/.jupysql/config.

Did not find user configurations in /Users/i.korsakov/_code/github/pet_project_what_is_ducklake/pyproject.toml.

In [3]:
%sql duckdb:///:memory:

Connecting and switching to connection 'duckdb:///:memory:'

# Создание подключения к DuckLake

In [4]:
%%sql
INSTALL ducklake;
INSTALL postgres;

,Success


In [5]:
%%sql
CREATE OR REPLACE SECRET
(
    TYPE postgres,
    HOST 'localhost',
    PORT 5432,
    DATABASE postgres,
    USER 'postgres',
    PASSWORD 'postgres'
);

,Success


In [6]:
%%sql
CREATE OR REPLACE SECRET
(
    TYPE s3,
    URL_STYLE 'path',
    USE_SSL FALSE,
    ENDPOINT 'localhost:9000',
    KEY_ID 'minioadmin',
    SECRET 'minioadmin'
);

,Success


In [7]:
%%sql
ATTACH 'ducklake:postgres:dbname=postgres' AS my_ducklake (DATA_PATH 's3://prod/ducklake/');

,Success


In [8]:
%%sql
USE my_ducklake;

,Success


# Создание таблицы в DuckLake

In [9]:
%%sql
INSTALL fakeit FROM community;
LOAD fakeit;

CREATE OR REPLACE TABLE fake_data AS
SELECT
    s.id AS id,
    fakeit_name_full() AS name,
    fakeit_contact_email() AS email,
    fakeit_address_city() AS city,
    fakeit_address_country() AS country
FROM
    generate_series(1, 100) AS s(id);

,Success


In [10]:
%%sql
FROM fake_data

,id,name,email,city,country
0,1,Akeem Conroy,felipelemke@kovacek.info,Hettingershire,Latvia
1,2,Audreanne Kerluke,leonelzboncak@bode.info,Gerardchester,Saudi Arabia
2,3,Elwin Borer,marcelinocasper@lehner.name,Windlershire,Brazil
3,4,Willy Luettgen,carmellalittle@rogahn.org,Reneeborough,Saint Vincent and the Grenadines
4,5,Americo Witting,seanlindgren@nitzsche.info,Nolanchester,Cayman Islands
...,...,...,...,...,...
95,96,Sid Schiller,averylebsack@olson.org,Carrieview,Macedonia
96,97,Lauren Dibbert,savionhudson@funk.name,Webershire,Yemen
97,98,Trevion Lesch,ryanwillms@pagac.biz,Jacyntheburgh,Malawi
98,99,Taurean Bruen,jadencasper@schumm.biz,Swaniawskiport,Kenya


# Изменение схемы (модели)
- [Schema Evolution](https://ducklake.select/docs/stable/duckdb/usage/schema_evolution)

In [11]:
%%sql
ALTER TABLE fake_data
ADD COLUMN name_prefix VARCHAR;

,Success


In [12]:
%%sql
from fake_data

,id,name,email,city,country,name_prefix
0,1,Akeem Conroy,felipelemke@kovacek.info,Hettingershire,Latvia,None
1,2,Audreanne Kerluke,leonelzboncak@bode.info,Gerardchester,Saudi Arabia,None
2,3,Elwin Borer,marcelinocasper@lehner.name,Windlershire,Brazil,None
3,4,Willy Luettgen,carmellalittle@rogahn.org,Reneeborough,Saint Vincent and the Grenadines,None
4,5,Americo Witting,seanlindgren@nitzsche.info,Nolanchester,Cayman Islands,None
...,...,...,...,...,...,...
95,96,Sid Schiller,averylebsack@olson.org,Carrieview,Macedonia,None
96,97,Lauren Dibbert,savionhudson@funk.name,Webershire,Yemen,None
97,98,Trevion Lesch,ryanwillms@pagac.biz,Jacyntheburgh,Malawi,None
98,99,Taurean Bruen,jadencasper@schumm.biz,Swaniawskiport,Kenya,None


In [13]:
%%sql
UPDATE fake_data
SET name_prefix = fakeit_name_prefix()

,Success


In [14]:
%%sql
from fake_data

,id,name,email,city,country,name_prefix
0,1,Akeem Conroy,felipelemke@kovacek.info,Hettingershire,Latvia,Mrs.
1,2,Audreanne Kerluke,leonelzboncak@bode.info,Gerardchester,Saudi Arabia,Ms.
2,3,Elwin Borer,marcelinocasper@lehner.name,Windlershire,Brazil,Miss
3,4,Willy Luettgen,carmellalittle@rogahn.org,Reneeborough,Saint Vincent and the Grenadines,Ms.
4,5,Americo Witting,seanlindgren@nitzsche.info,Nolanchester,Cayman Islands,Mr.
...,...,...,...,...,...,...
95,96,Sid Schiller,averylebsack@olson.org,Carrieview,Macedonia,Mr.
96,97,Lauren Dibbert,savionhudson@funk.name,Webershire,Yemen,Mr.
97,98,Trevion Lesch,ryanwillms@pagac.biz,Jacyntheburgh,Malawi,Mr.
98,99,Taurean Bruen,jadencasper@schumm.biz,Swaniawskiport,Kenya,Dr.


# Time travel

In [15]:
%%sql
SELECT current_catalog()

,current_catalog()
0,my_ducklake


In [16]:
%%sql
USE '__ducklake_metadata_my_ducklake'

,Success


In [17]:
%%sql
FROM ducklake_snapshot_changes

,snapshot_id,changes_made,author,commit_message,commit_extra_info
0,0,"created_schema:""main""",None,None,None
1,1,"created_table:""main"".""fake_data"",inserted_into...",None,None,None
2,2,altered_table:1,None,None,None
3,3,"inserted_into_table:1,deleted_from_table:1",None,None,None


In [18]:
%%sql
FROM ducklake_snapshot

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,0,2026-06-01 13:46:00.504489+03:00,0,1,0
1,1,2026-06-01 13:46:01.143185+03:00,1,2,1
2,2,2026-06-01 13:46:01.374568+03:00,2,2,1
3,3,2026-06-01 13:46:02.637690+03:00,2,2,2


In [19]:
%%sql
USE 'my_ducklake';

,Success


In [21]:
%%sql
SELECT * FROM fake_data AT (VERSION => 1);

,id,name,email,city,country
0,1,Akeem Conroy,felipelemke@kovacek.info,Hettingershire,Latvia
1,2,Audreanne Kerluke,leonelzboncak@bode.info,Gerardchester,Saudi Arabia
2,3,Elwin Borer,marcelinocasper@lehner.name,Windlershire,Brazil
3,4,Willy Luettgen,carmellalittle@rogahn.org,Reneeborough,Saint Vincent and the Grenadines
4,5,Americo Witting,seanlindgren@nitzsche.info,Nolanchester,Cayman Islands
...,...,...,...,...,...
95,96,Sid Schiller,averylebsack@olson.org,Carrieview,Macedonia
96,97,Lauren Dibbert,savionhudson@funk.name,Webershire,Yemen
97,98,Trevion Lesch,ryanwillms@pagac.biz,Jacyntheburgh,Malawi
98,99,Taurean Bruen,jadencasper@schumm.biz,Swaniawskiport,Kenya
